In [ ]:
import pandas as pd

# Lists of questions of each institution: labor law questions with answer (the annulled ones are already out)
INPUT_PATHS = {
    'Tribunal Regional do Trabalho': 'csvs/trt/trt_questions_all_bench.csv',
    'Ordem dos Advogados do Brasil': 'csvs/oab/oab_questions_with_answer.csv',
    'Ministério do Trabalho e Emprego': 'csvs/mte/mte_questions_all.csv',
}
COLUMNS = ['question_number', 'question', 'options', 'exam', 'answer']

In [ ]:
dfs = []
for institution, path in INPUT_PATHS.items():
    df = pd.read_csv(path, encoding='utf-8-sig')[COLUMNS]
    df['institution'] = institution
    print(f'{path}: {len(df)} questions in {df["exam"].nunique()} exams')
    dfs.append(df)

In [ ]:
questions = pd.concat(dfs, ignore_index=True)
print(f'{len(questions)} questions in {questions["exam"].nunique()} exams')
print('Questions without answer:', questions['answer'].isna().sum())
questions.groupby('institution').agg(questions=('question', 'size'), exams=('exam', 'nunique'))

In [ ]:
# Duplicated questions: the same statement and the same options (in the same order). The first one is kept
duplicated = questions.duplicated(['question', 'options'])
removed = questions[duplicated]
different_answer = questions[questions.duplicated(['question', 'options'], keep=False)].groupby(['question', 'options'])['answer'].nunique() > 1
print(f'Duplicated questions removed: {len(removed)}')
print(f'Duplicated questions with different answers: {different_answer.sum()}')
for row in removed.itertuples():
    print(f'  {row.exam}: {row.question_number}')
questions = questions[~duplicated].reset_index(drop=True)
print(f'{len(questions)} questions in {questions["exam"].nunique()} exams')

In [ ]:
import ast
import re

OPTION_LETTER = re.compile(r'\([A-E]\)')


def question_problems(row):
    # Failures of the extraction of a question; [] when the question is complete. OAB questions have 4 options,
    # the others 5
    options = ast.literal_eval(row['options'])
    n_options = 4 if ' - OAB - ' in row['exam'] else 5
    problems = []
    if not row['question'].strip():
        problems.append('empty statement')
    if len(options) != n_options:
        problems.append(f'{len(options)} options instead of {n_options}')
    if any(not option.strip() for option in options):
        problems.append('empty option')
    # Option letters left in the text: the options were not split from the statement ('(A) (B) (C) (D) (E)')
    if OPTION_LETTER.search(row['question']) or any(OPTION_LETTER.search(option) for option in options):
        problems.append('option letter in the text')
    if row['answer'] not in 'ABCDE'[:len(options)]:
        problems.append(f'answer {row["answer"]} is not an option')
    return problems


problems = questions.apply(question_problems, axis=1)
failed = questions[problems.str.len() > 0]
print(f'Questions with failures removed: {len(failed)}')
for row, row_problems in zip(failed.itertuples(), problems[failed.index]):
    print(f'  {row.exam}: {row.question_number} ({"; ".join(row_problems)})')
questions = questions[problems.str.len() == 0].reset_index(drop=True)
print(f'{len(questions)} questions in {questions["exam"].nunique()} exams')

In [ ]:
import json
import os

# Board of each TRT exam: prefix of the exam file name ('fcc-2011-trt-1-regiao-...', 'fgv-2017-trt-12-...')
with open('extraction_config/trt_exams.json', encoding='utf-8') as f:
    trt_boards = {exam['exam']: os.path.basename(exam['prova_path']).split('-')[0].upper() for exam in json.load(f)}

# The same TRT positions written in different ways in the exam names
POSITIONS = {
    'Analista Judiciário - Área Judiciária - Especialidade Oficial de Justiça Avaliador Federal': 'Analista Judiciário - Oficial de Justiça Avaliador Federal',
    'Analista Judiciário - Área Judiciária - Oficial de Justiça Avaliador Federal': 'Analista Judiciário - Oficial de Justiça Avaliador Federal',
    'Analista Judiciário - Especialidade Oficial de Justiça Avaliador Federal': 'Analista Judiciário - Oficial de Justiça Avaliador Federal',
    'Analista Judiciário - Oficial de Justiça Avaliador': 'Analista Judiciário - Oficial de Justiça Avaliador Federal',
    'Analista Judiciário - Área Judiciária - Execução de Mandados': 'Analista Judiciário - Execução de Mandados',
}


def exam_info(exam):
    # (board, position, institution, exam_type) from the exam name
    if ' - TRT ' in exam:
        # '{position} - TRT {region}ª Região - {year}'
        position, rest = exam.split(' - TRT ')
        region = rest.split(' Região')[0]
        return trt_boards[exam], POSITIONS.get(position, position), f'Tribunal Regional do Trabalho {region} Região', 'public_service'
    if ' - OAB - ' in exam:
        board = 'CESPE/CEBRASPE' if exam.startswith('CESPE') else 'FGV'
        return board, 'OAB', 'Ordem dos Advogados do Brasil', 'bar_exam'
    # MTE: ESAF exams and Curso de Formação (CEBRASPE)
    board = 'CESPE/CEBRASPE' if exam.startswith('Curso de Formação') else 'ESAF'
    return board, 'Auditor-Fiscal do Trabalho', 'Ministério do Trabalho e Emprego', 'public_service'


questions['year'] = questions['exam'].str.extract(r'\b((?:19|20)\d{2})\b', expand=False).astype(int)
questions[['board', 'position', 'institution', 'exam_type']] = questions['exam'].apply(exam_info).tolist()
questions.groupby(['institution', 'board', 'exam_type']).size()

In [ ]:
import ast
import tiktoken

# Options as a JSON object {"A": ..., "B": ...}. Words (whitespace tokenization) and tokens (o200k_base
# tokenizer) of the statement and of the options: the options are counted one by one, without the letters
encoding = tiktoken.get_encoding('o200k_base')
count_words = lambda text: len(text.split())
count_tokens = lambda text: len(encoding.encode(text))
options = questions['options'].apply(ast.literal_eval)
questions['options'] = options.apply(lambda o: json.dumps(dict(zip('ABCDE', o)), ensure_ascii=False))
questions['num_options'] = options.apply(len)
questions['num_words'] = questions['question'].apply(count_words)
questions['num_tokens'] = questions['question'].apply(count_tokens)
questions['num_words_options'] = options.apply(lambda o: sum(count_words(option) for option in o))
questions['num_tokens_options'] = options.apply(lambda o: sum(count_tokens(option) for option in o))
questions['num_words_total'] = questions['num_words'] + questions['num_words_options']
questions['num_tokens_total'] = questions['num_tokens'] + questions['num_tokens_options']
questions[['num_options', 'num_words', 'num_tokens', 'num_words_options', 'num_tokens_options', 'num_words_total', 'num_tokens_total']].describe()

In [ ]:
# Order of the dataset: TRT, OAB and MTE; inside each one, the most recent exams first and the questions in
# exam order. The id follows this order. The columns are the ones of the data dictionary, in its order
OUTPUT_COLUMNS = [
    'id', 'question_number', 'question', 'options', 'answer', 'exam', 'year', 'board', 'position',
    'num_options', 'num_words', 'num_tokens', 'institution', 'exam_type',
    'num_words_options', 'num_words_total', 'num_tokens_options', 'num_tokens_total',
]
group = questions['institution'].map(lambda i: 0 if i.startswith('Tribunal') else 1 if i.startswith('Ordem') else 2)
questions = questions.assign(group=group).sort_values(
    ['group', 'year', 'exam', 'question_number'], ascending=[True, False, True, True], ignore_index=True)
questions['id'] = [f'CLTBench-{k:04d}' for k in range(1, len(questions) + 1)]
questions = questions[OUTPUT_COLUMNS]
questions.head()

In [ ]:
OUTPUT_PATH = 'cltbench.csv'
questions.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')
print(f'{OUTPUT_PATH}: {len(questions)} questions in {questions["exam"].nunique()} exams')

# JSON Lines: one question per line; the options are a JSON object (in the CSV they are a JSON string)
OUTPUT_JSONL_PATH = 'cltbench.jsonl'
with open(OUTPUT_JSONL_PATH, 'w', encoding='utf-8') as f:
    for record in questions.to_dict(orient='records'):
        record['options'] = json.loads(record['options'])
        f.write(json.dumps(record, ensure_ascii=False) + '\n')
print(f'{OUTPUT_JSONL_PATH}: {len(questions)} questions')